# TBBT - profile postaci (E6)

Buduje pięć profili tożsamości: wybiera po osiem wzorcowych twarzy na postać i liczy ich margines rozstrzygalności. Wybór przykładów jest ręczny i odbywa się między dwoma uruchomieniami notatnika.

**Wymaga:** `tbbt_03_annotations.ipynb` (rejestr i zakresy korpusu muszą istnieć) oraz ręcznego wypełnienia `data/annotations/tbbt/tbbt_profiles.csv` po sekcji 3.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `data/cache/faces/crops/tbbt/*.npz` | bufor detekcji RetinaFace na odcinek |
| `data/cache/faces/arcface/tbbt/*.npz` | wektory ArcFace na odcinek |
| `data/interim/tbbt/work/profile_candidates/*.png` | kontaktówki miniatur twarzy do przejrzenia ręcznie |
| `data/interim/tbbt/work/profile_candidates/<postac>_final.png` | podgląd dokładnie wybranych 8 przykładów danej postaci |
| `data/annotations/tbbt/tbbt_profiles.csv` | wskaźniki wybranych przykładów; praca ręczna, notatnik dopisuje tylko nagłówek, gdy pliku jeszcze nie ma |
| `data/cache/faces/profiles/tbbt.npz` | zbudowane macierze profilu i margines na postać |

**Dalej:** eksperymenty, `notebooks/README.md` (E6).

In [ ]:
SERIES = "tbbt"

import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.annotation import profiles
from src.annotation import registry as reg
from src.data import ranges as rg
from src.features import faces, identity
from src.utils import settings

for module in (profiles, reg, rg, faces, identity, settings):
    importlib.reload(module)

ANNOTATIONS_DIR = ROOT / "data" / "annotations" / SERIES
WORK_DIR        = ROOT / "data" / "interim" / SERIES / "work"
CANDIDATES_DIR  = WORK_DIR / "profile_candidates"
REGISTER_CSV    = ROOT / "data" / "interim" / SERIES / f"{SERIES}_annotations.csv"

PROFILES_CSV = profiles.path_for(SERIES, ANNOTATIONS_DIR)
CHARACTERS   = settings.profiled(SERIES)

episodes = rg.split_episodes(rg.load_ranges(SERIES), "dev")
register_rows = (reg.accepted(reg.load(REGISTER_CSV), split="dev")
                 if REGISTER_CSV.exists() else [])

print(f"{SERIES}: {len(episodes)} dev episodes, {len(register_rows)} accepted dev descriptions")
print(f"profiled characters: {', '.join(CHARACTERS)}")
print(f"profile file: {PROFILES_CSV.relative_to(ROOT)}"
      + ("" if PROFILES_CSV.exists() else " (not created yet - see section 3)"))

## 1. Bufor twarzy i wektory tożsamości

**Zapisuje:** `data/cache/faces/crops/tbbt/*.npz` (detekcja i wyrównanie RetinaFace) oraz `data/cache/faces/arcface/tbbt/*.npz` (wektory ArcFace) - pliki pamięci podręcznej, odcinek po odcinku, nadpisywane tylko przy `force=True`. Jeśli bufor TBBT jeszcze nie istnieje, ten blok uruchamia RetinaFace nad odcinkami dev po raz pierwszy i potrwa dłużej niż przy *The Office*.

In [ ]:
covered = faces.ensure_faces(SERIES, episodes, log=print)
identity.ensure_identity(SERIES, episodes, log=print)

missing = set(episodes) - set(covered)
if missing:
    print(f"\nno face buffer yet for: {', '.join(sorted(missing))}")

## 2. Kontaktówki kandydatów

**Zapisuje:** `data/interim/tbbt/work/profile_candidates/<postac>_<odcinek>.png` - siatki miniatur `112x112`, podpisane `sekundy (mm:ss)  #face_index  score`, ograniczone do okien czasu, w których opis danego odcinka wzmiankuje imię postaci (`profiles.candidate_windows`, margines `MARGIN` sekund wokół wzmianki). Blok tylko rysuje kandydatów; kto jest kim, rozstrzyga się w sekcji 3.

Ten sam blok dopisuje nagłówek `tbbt_profiles.csv`, jeśli plik jeszcze nie istnieje (`profiles.ensure_skeleton`), i nigdy nie rusza go, jeśli już tam coś jest.

In [ ]:
MARGIN = 10.0   # seconds around a mention of the character's name
LIMIT  = 48     # max thumbnails per sheet -- a busy episode can mention someone
                # dozens of times and match a thousand faces, more than anyone
                # would look at; shown ones are spread evenly across the matches

created = profiles.ensure_skeleton(PROFILES_CSV)
print(f"{'created empty' if created else 'kept existing'} {PROFILES_CSV.relative_to(ROOT)}")

written = []
for character in CHARACTERS:
    windows_by_episode = profiles.candidate_windows(register_rows, character, margin=MARGIN)
    for episode, windows in sorted(windows_by_episode.items()):
        if episode not in episodes:
            continue
        result = profiles.contact_sheet(
            SERIES, episode, windows, CANDIDATES_DIR / f"{character}_{episode}.png",
            limit=LIMIT)
        if result:
            written.append({"character": character, "episode": episode, **result})

print(f"\n{len(written)} contact sheets -> {CANDIDATES_DIR.relative_to(ROOT)}")
for character in CHARACTERS:
    sheets = [w for w in written if w["character"] == character]
    if not sheets:
        print(f"  {character:<10}WARNING: no mention in any dev episode's description")
        continue
    total_matched = sum(s["matched"] for s in sheets)
    total_shown = sum(s["shown"] for s in sheets)
    print(f"  {character:<10}{len(sheets)} episode(s), {total_shown}/{total_matched}"
          " faces shown")

## 3. Przerwa: wybór ręczny

Otworzyć pliki z `data/interim/tbbt/work/profile_candidates/` i dla każdej z 5 postaci wybrać 8 dobrych przykładów: różne ujęcia, kąty, oświetlenie, sezony. Wpisać je do `data/annotations/tbbt/tbbt_profiles.csv`:

```
character;episode;time;face_index;note
Sheldon;s01e15;123.40;0;
```

`character` to samo imię, dokładnie jak w `CHARACTERS` (`Sheldon`, `Leonard`, `Penny`, `Howard`, `Raj`), bez nazwiska. Podpis miniatury ma postać `sekundy (mm:ss)  #face_index  score` - do `time` przepisać sekundy, zegar w nawiasie jest tylko do sprawdzenia w odtwarzaczu. Jeśli w kontaktówkach danej postaci brakuje dobrych przykładów, można poszerzyć `MARGIN` w sekcji 2 i uruchomić ją ponownie; to odświeży pliki `.png`, a `tbbt_profiles.csv` zostaje nietknięty.

Po zapisaniu pliku wrócić i uruchomić notatnik od sekcji 4.

## 4. Podgląd wybranych przykładów

**Zapisuje:** `data/interim/tbbt/work/profile_candidates/<postac>_final.png` - siatka dokładnie tych 8 przykładów, które są teraz w `tbbt_profiles.csv`, podpisana notatką z pliku.

Sprawdzenie wzrokiem, że wpisane wskaźniki pokazują właściwą osobę, zanim policzy się kontrola numeryczna w sekcji 5. Wskaźnik może trafić w kogoś innego stojącego obok, a to nie wyjdzie z samych liczb tak łatwo, jak z jednego spojrzenia na twarz.

In [ ]:
preview_rows = identity.load_profile_rows(SERIES)

if not preview_rows:
    print(f"{PROFILES_CSV.name} is empty - fill it in first (section 3)")
else:
    for character in CHARACTERS:
        count = sum(1 for r in preview_rows if r["character"] == character)
        out_path = CANDIDATES_DIR / f"{character}_final.png"
        result = profiles.final_contact_sheet(SERIES, character, preview_rows, out_path)
        if result:
            print(f"  {character:<10}{count} example(s) -> {out_path.relative_to(ROOT)}")
        else:
            print(f"  {character:<10}no rows yet")

## 5. Kontrola jakości

Uruchomić po ręcznym wypełnieniu `tbbt_profiles.csv`. Margines na postać to średnie podobieństwo do własnych przykładów minus najlepsze podobieństwo do przykładu innej postaci. Niżej to samo na poziomie pojedynczego przykładu (`own_similarity`, `best_other`), żeby widać było, który z 8 wierszy psuje margines, a nie tylko że któryś go psuje.

Przykład z niskim wkładem (`own_similarity - best_other`) albo oznaczony `problem` (wskaźnik poza zakresem, czyli zły `time` lub `face_index`) trzeba podmienić w pliku z sekcji 3 i wrócić tutaj, albo do sekcji 4, żeby zobaczyć twarz od razu.

In [ ]:
profile_rows = identity.load_profile_rows(SERIES)
dev_episodes = set(episodes)

if not profile_rows:
    print(f"{PROFILES_CSV.name} is empty - fill it in first (section 3)")
else:
    report = profiles.quality_report(SERIES, profile_rows, dev_episodes)

    header = f"{'character':<10}{'examples':>9}{'episodes':>9}{'margin':>9}"
    print(header)
    print("-" * len(header))
    for character in CHARACTERS:
        rows_here = [r for r in profile_rows if r["character"] == character]
        n_episodes = len({r["episode"] for r in rows_here})
        margin = report["margin"].get(character, float("nan"))
        flag = "" if len(rows_here) == identity.PROFILE_SIZE else "  != PROFILE_SIZE"
        print(f"{character:<10}{len(rows_here):>9}{n_episodes:>9}{margin:>9.3f}{flag}")

    print("\nexamples worth a second look (contribution < 0.10, or a flagged pointer):")
    for example in report["examples"]:
        contribution = (None if example["own_similarity"] is None
                        else example["own_similarity"] - example["best_other"])
        if example["problem"] or contribution is None or contribution < 0.10:
            reason = example["problem"] or f"contribution {contribution:.3f}"
            print(f"  {example['character']:<10}{example['episode']:<10}"
                  f"{example['time']:>10}  #{example['face_index']}  {reason}")

## 6. Budowa i bufor profilu

**Zapisuje:** `data/cache/faces/profiles/tbbt.npz` - macierze przykładów i margines na postać. Czyta to `src/runners/components.py` przy włączonym komponencie `identity` (E6).

Uruchomić dopiero gdy sekcja 5 nie zgłasza już żadnych przykładów do podmiany; `build_profiles` i tak odrzuci wiersz spoza odcinków dev.

In [ ]:
built = identity.build_profiles(SERIES, dev_episodes, log=print)
saved = identity.save_profiles(SERIES, built)

print(f"saved -> {saved.relative_to(ROOT)}")
for character in CHARACTERS:
    if character not in built["characters"]:
        print(f"  WARNING: no examples for {character} yet - fill in section 3")